# Aegis-AI Training Notebook

This Colab notebook is written to be stable on modern Google Colab runtimes. It downloads MalImg from Kaggle, saves outputs to Google Drive, trains EfficientNet-B1 with mixed precision, trains the sklearn head, computes the Fisher matrix, prepares replay/validation assets, and exports ONNX.

Important: MalImg already contains malware images. You are not training on live malware binaries here.

## 1. Mount Google Drive
Drive is used for checkpoints, logs, and exported artifacts so you do not lose them if the Colab session resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Kaggle dataset setup
Upload `kaggle.json`, then the notebook downloads the MalImg dataset directly inside Colab.

In [ ]:
from google.colab import files
files.upload()


In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


## 3. Install only the extra packages we actually need

This cell deliberately avoids forcing NumPy or pandas versions, because changing those on Colab often breaks the runtime. If Colab asks for a restart after this cell, restart the session and rerun from the top.

In [ ]:
%pip install -q kaggle mlflow timm onnx onnxruntime joblib


In [ ]:
!kaggle datasets download -d ikrambenabd/malimg-original
!unzip -o malimg-original.zip


In [ ]:
import os

print(os.listdir('/content/malimg_paper_dataset_imgs')[:5])
print('Classes:', len(os.listdir('/content/malimg_paper_dataset_imgs')))


## 4. Imports and version check

In [ ]:
from pathlib import Path
import json
import os
import pickle
from collections import defaultdict

import joblib
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt

print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('mlflow:', mlflow.__version__)
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## 5. Project configuration

In [ ]:
CLASS_LABELS = [
    'Adialer.C','Agent.FYI','Allaple.A','Allaple.L','Alueron.gen!J','Autorun.K',
    'C2LOP.gen!g','C2LOP.P','Dialplatform.B','Dontovo.A','Fakerean','Instantaccess',
    'Lolyda.AA1','Lolyda.AA2','Lolyda.AA3','Lolyda.AT','Malex.gen!J','Obfuscator.AD',
    'Rbot!gen','Skintrim.N','Swizzor.gen!E','Swizzor.gen!I','VB.AT','Wintrim.BX','Yuner.A','Benign'
]
DATASET_ROOT = Path('/content/malimg_paper_dataset_imgs')
OUTPUT_ROOT = Path('/content/drive/MyDrive/AegisAI/outputs')
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
ARTIFACT_DIR = OUTPUT_ROOT / 'artifacts'
MLFLOW_DIR = OUTPUT_ROOT / 'mlruns'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f'file:{MLFLOW_DIR}')
mlflow.set_experiment('aegis-ai-malimg')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 16
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4
NUM_WORKERS = 2


## 6. Dataset discovery and splitting

In [ ]:
all_records = []
for family_dir in sorted(DATASET_ROOT.iterdir()):
    if family_dir.is_dir() and family_dir.name in CLASS_LABELS:
        for image_path in family_dir.glob('*.png'):
            all_records.append((image_path, family_dir.name))

labels = [label for _, label in all_records]
train_records, temp_records = train_test_split(
    all_records, test_size=0.2, random_state=42, stratify=labels
)
temp_labels = [label for _, label in temp_records]
val_records, test_records = train_test_split(
    temp_records, test_size=0.5, random_state=42, stratify=temp_labels
)

print('Train:', len(train_records))
print('Validation:', len(val_records))
print('Test:', len(test_records))


In [ ]:
class MalImgDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        path, label = self.records[idx]
        image = Image.open(path).convert('RGB')
        image = self.transform(image)
        return image, CLASS_LABELS.index(label)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_loader = DataLoader(MalImgDataset(train_records, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(MalImgDataset(val_records, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(MalImgDataset(test_records, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())


## 7. Model, optimizer, and AMP setup

In [ ]:
model = timm.create_model('efficientnet_b1', pretrained=True, num_classes=len(CLASS_LABELS)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
best_val_acc = 0.0
best_checkpoint_path = CHECKPOINT_DIR / 'efficientnet_b1_best.pth'


In [ ]:
def evaluate_model(loader):
    model.eval()
    all_preds, all_truths = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(images)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_truths.extend(labels.cpu().numpy().tolist())
    return accuracy_score(all_truths, all_preds), all_preds, all_truths


## 8. Training loop

In [ ]:
with mlflow.start_run(run_name='initial_training'):
    mlflow.log_param('batch_size', BATCH_SIZE)
    mlflow.log_param('epochs', NUM_EPOCHS)
    mlflow.log_param('learning_rate', LEARNING_RATE)

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.item())

        train_loss = running_loss / max(len(train_loader), 1)
        val_acc, _, _ = evaluate_model(val_loader)

        mlflow.log_metric('train_loss', train_loss, step=epoch + 1)
        mlflow.log_metric('val_accuracy', val_acc, step=epoch + 1)

        checkpoint_payload = {
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_accuracy': val_acc,
        }
        torch.save(checkpoint_payload, CHECKPOINT_DIR / f'checkpoint_epoch_{epoch + 1}.pth')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_checkpoint_path)

        print(f'Epoch {epoch + 1}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | val_acc={val_acc:.4f}')

    mlflow.log_metric('best_val_accuracy', best_val_acc)


## 9. Load best model and create embedding extractor

In [ ]:
model.load_state_dict(torch.load(best_checkpoint_path, map_location=DEVICE))
model.eval()

class EmbeddingModel(nn.Module):
    def __init__(self, inner_model):
        super().__init__()
        self.inner_model = inner_model

    def forward(self, x):
        features = self.inner_model.forward_features(x)
        pooled = self.inner_model.global_pool(features)
        if pooled.ndim > 2:
            pooled = torch.flatten(pooled, 1)
        return pooled

embedding_model = EmbeddingModel(model).to(DEVICE)
embedding_model.eval()


In [ ]:
def collect_embeddings(loader):
    embeddings, labels = [], []
    with torch.no_grad():
        for images, y in loader:
            images = images.to(DEVICE, non_blocking=True)
            emb = embedding_model(images)
            embeddings.append(emb.cpu().numpy())
            labels.append(y.numpy())
    return np.concatenate(embeddings), np.concatenate(labels)

train_emb, train_y = collect_embeddings(train_loader)
test_emb, test_y = collect_embeddings(test_loader)
print(train_emb.shape, test_emb.shape)


## 10. Train sklearn classification head

In [ ]:
head = LogisticRegression(max_iter=2000, multi_class='multinomial')
head.fit(train_emb, train_y)
test_pred = head.predict(test_emb)
head_accuracy = accuracy_score(test_y, test_pred)
print('Head accuracy:', head_accuracy)
joblib.dump(head, ARTIFACT_DIR / 'head.pkl')


In [ ]:
cm = confusion_matrix(test_y, test_pred)
plt.figure(figsize=(14, 10))
sns.heatmap(cm, cmap='mako')
plt.title('MalImg Test Confusion Matrix')
plt.show()
print(classification_report(test_y, test_pred, target_names=CLASS_LABELS))


## 11. Compute Fisher matrix for EWC

In [ ]:
fisher = {name: torch.zeros_like(param, device=DEVICE) for name, param in model.named_parameters()}
model.eval()
for images, labels in train_loader:
    images = images.to(DEVICE, non_blocking=True)
    labels = labels.to(DEVICE, non_blocking=True)
    model.zero_grad(set_to_none=True)
    logits = model(images)
    loss = criterion(logits, labels)
    loss.backward()

    for name, param in model.named_parameters():
        if param.grad is not None:
            fisher[name] += param.grad.detach().pow(2)

for name in fisher:
    fisher[name] /= max(len(train_loader), 1)

with open(ARTIFACT_DIR / 'fisher_matrix.pkl', 'wb') as handle:
    pickle.dump({name: tensor.detach().cpu() for name, tensor in fisher.items()}, handle)


## 12. Build replay buffer and validation set

In [ ]:
def image_to_array(path):
    image = Image.open(path).convert('RGB').resize((224, 224))
    array = np.asarray(image).astype(np.float32) / 255.0
    return np.transpose(array, (2, 0, 1))

replay_buffer = []
per_family_counts = defaultdict(int)
for image_path, label in train_records:
    if per_family_counts[label] >= 8:
        continue
    replay_buffer.append({
        'image': image_to_array(image_path).tolist(),
        'label': label,
        'label_index': CLASS_LABELS.index(label),
        'path': str(image_path),
    })
    per_family_counts[label] += 1
    if sum(per_family_counts.values()) >= 200:
        break

with open(ARTIFACT_DIR / 'replay_buffer.pkl', 'wb') as handle:
    pickle.dump(replay_buffer, handle)

validation_set = []
validation_counts = defaultdict(int)
for image_path, label in val_records + test_records:
    if validation_counts[label] >= 100:
        continue
    validation_set.append({
        'image': image_to_array(image_path).tolist(),
        'label': label,
        'label_index': CLASS_LABELS.index(label),
        'path': str(image_path),
    })
    validation_counts[label] += 1

with open(ARTIFACT_DIR / 'validation_set.pkl', 'wb') as handle:
    pickle.dump(validation_set, handle)

print('Replay buffer size:', len(replay_buffer))
print('Validation set size:', len(validation_set))


## 13. Export ONNX and save artifacts

In [ ]:
dummy = torch.randn(1, 3, 224, 224, device=DEVICE)
onnx_path = ARTIFACT_DIR / 'efficientnet_b1.onnx'
torch.onnx.export(
    embedding_model,
    dummy,
    str(onnx_path),
    input_names=['input'],
    output_names=['embedding'],
    dynamic_axes={'input': {0: 'batch_size'}, 'embedding': {0: 'batch_size'}},
    opset_version=17,
)
torch.save(model.state_dict(), ARTIFACT_DIR / 'efficientnet_b1_pytorch.pth')
print('Saved ONNX to:', onnx_path)


In [ ]:
%pip install -q onnxscript # move it above the above cell

## 14. Final summary

After the notebook finishes, copy these files from `MyDrive/AegisAI/outputs/artifacts/` into your local project:

- `efficientnet_b1.onnx`
- `efficientnet_b1_pytorch.pth`
- `head.pkl`
- `fisher_matrix.pkl`
- `replay_buffer.pkl`
- `validation_set.pkl`
